In [50]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re
from datetime import datetime
from collections import deque


In [51]:
def scrape_bbc_urdu_improved(target_count=250, max_depth=3):
    """
    Improved scraper with recursive discovery to collect 220+ unique articles
    Uses BFS (Breadth-First Search) to discover articles from links within articles
    """
    articles_data = {}
    article_bodies = {}
    article_num = 1
    visited_urls = set()
    seen_titles = set()  
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
  
    url_queue = deque()
    

    seed_urls = [
        'https://www.bbc.com/urdu',
        'https://www.bbc.com/urdu/pakistan',
        'https://www.bbc.com/urdu/world',
        'https://www.bbc.com/urdu/india',
        'https://www.bbc.com/urdu/science',
        'https://www.bbc.com/urdu/entertainment',
        'https://www.bbc.com/urdu/sports',
        'https://www.bbc.com/urdu/topics/cp7r8vgl2fmt',  # Latest news
        'https://www.bbc.com/urdu/topics/c340q430z4vt',  # Pakistan
        'https://www.bbc.com/urdu/topics/c2dwqd2gy93t',  # International
        'https://www.bbc.com/urdu/topics/c06gq9v4xj3t',  # India
        'https://www.bbc.com/urdu/topics/c2lej05jnm3t',  # Entertainment
        'https://www.bbc.com/urdu/topics/cr50y580rjxt',  # Sports
        'https://www.bbc.com/urdu/topics/c4794229y0nt',  # Science
        'https://www.bbc.com/urdu/topics/ckdxnw959n7t',  # Health
    ]
    
    for url in seed_urls:
        url_queue.append((url, 0))
        visited_urls.add(url)
    
    print(f"Starting to scrape BBC Urdu articles (target: {target_count})...")
    print(f"Using recursive discovery with max depth: {max_depth}")
    print("=" * 70)
    
    while url_queue and article_num <= target_count:
        current_url, depth = url_queue.popleft()
        
        # Don't go too deep
        if depth > max_depth:
            continue
        
        try:
            print(f"\n[Depth {depth}] Scanning: {current_url[:70]}...")
            response = requests.get(current_url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"  Skipping (Status {response.status_code})")
                continue
                
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all links on this page
            all_links = soup.find_all('a', href=True)
            article_links = []
            new_discovery_links = []
            
            for link in all_links:
                href = link['href']
                
                # Build full URL
                if href.startswith('http'):
                    full_url = href
                elif href.startswith('/'):
                    full_url = f"https://www.bbc.com{href}"
                else:
                    continue
                
                # Only process BBC Urdu links
                if '/urdu/' not in full_url:
                    continue
                
                # Skip if already visited
                if full_url in visited_urls:
                    continue
                
                visited_urls.add(full_url)
                
                # Check if it's an article link
                if (('/urdu/articles/' in full_url) or 
                    (re.match(r'https://www\.bbc\.com/urdu/[a-z]+-\d+', full_url))):
                    article_links.append(full_url)
                # Check if it's a category or listing page (for discovery)
                elif depth < max_depth and any(cat in full_url for cat in ['pakistan', 'world', 'india', 'entertainment', 'sports', 'science', 'topics']):
                    new_discovery_links.append(full_url)
            
            print(f"  Found {len(article_links)} new articles, {len(new_discovery_links)} discovery pages")
            
            # Add discovery pages to queue
            for disc_url in new_discovery_links:
                url_queue.append((disc_url, depth + 1))
            
            # Scrape each article
            for article_url in article_links:
                if article_num > target_count:
                    break
                
                try:
                    # Get article page
                    time.sleep(0.2)  # Be respectful but fast
                    article_response = requests.get(article_url, headers=headers, timeout=15)
                    article_soup = BeautifulSoup(article_response.content, 'html.parser')
                    
                    # Extract title
                    title = None
                    title_tag = article_soup.find('h1')
                    if title_tag:
                        title = title_tag.get_text(strip=True)
                    
                    if not title or len(title) < 5:
                        continue
                    
                    # Check for duplicate title
                    if title in seen_titles:
                        continue
                    
                    seen_titles.add(title)
                    
                    # Extract date
                    date = datetime.now().strftime('%Y-%m-%d')
                    time_tag = article_soup.find('time')
                    if time_tag and 'datetime' in time_tag.attrs:
                        date = time_tag['datetime'][:10]
                    
                    # Extract article body
                    paragraphs = article_soup.find_all('p')
                    body_parts = []
                    
                    for p in paragraphs:
                        text = p.get_text(strip=True)
                        if text and len(text) > 30:
                            # Check if it contains Urdu text
                            if re.search(r'[\u0600-\u06FF]', text):
                                body_parts.append(text)
                    
                    body_text = ' '.join(body_parts)
                    
                    # Only save if we have substantial content
                    if len(body_text) > 300:
                        articles_data[str(article_num)] = {
                            "title": title,
                            "publish_date": date
                        }
                        article_bodies[str(article_num)] = body_text
                        
                        print(f"  ✓ [{article_num}] {title[:55]}...")
                        article_num += 1
                        
                        # Extract links from article for more discovery (increased from 10 to 30)
                        if depth < max_depth:
                            article_links_in_page = article_soup.find_all('a', href=True)
                            for a_link in article_links_in_page[:30]:  # Increased limit
                                href = a_link.get('href', '')
                                # More permissive URL matching
                                if ('/urdu/articles' in href or  
                                    '/urdu/pakistan' in href or
                                    '/urdu/world' in href or
                                    '/urdu/india' in href or
                                    '/urdu/entertainment' in href or
                                    '/urdu/sports' in href or
                                    '/urdu/science' in href or
                                    re.match(r'/urdu/[a-z]+-\d+', href)):
                                    full_url = f"https://www.bbc.com{href}" if href.startswith('/') else href
                                    if full_url not in visited_urls and '/urdu/' in full_url:
                                        url_queue.append((full_url, depth + 1))
                                        visited_urls.add(full_url)
                    
                except Exception as e:
                    continue
            
        except Exception as e:
            print(f"  Error: {str(e)[:50]}")
            continue
    
    print("\n" + "=" * 70)
    print(f"Successfully scraped {len(articles_data)} UNIQUE articles")
    print(f"Total URLs visited: {len(visited_urls)}")
    return articles_data, article_bodies


In [ ]:
# ============================================================================
# SECTION 2: TEXT PREPROCESSING
# ============================================================================

import re
import unicodedata

# ── Urdu diacritics (harakat + tatweel) ─────────────────────────────────────
URDU_DIACRITICS = [
    '\u064B', '\u064C', '\u064D', '\u064E', '\u064F',  
    '\u0650', '\u0651', '\u0652', '\u0653', '\u0654',
    '\u0655', '\u0656', '\u0657', '\u0658', '\u0670',  
    '\u0640',                                          
]

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)

def remove_diacritics(text):
    for diacritic in URDU_DIACRITICS:
        text = text.replace(diacritic, '')
    return text

def remove_urls(text):
    text = re.sub(
        r'http[s]?://(?:[a-zA-Z0-9$\-_@.&+!*(),]|(?:%[0-9a-fA-F]{2}))+',
        '', text
    )
    text = re.sub(r'www\.[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', '', text)
    return text

def remove_emojis(text):
    """Remove emoji and pictographic symbols."""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   
        "\U0001F300-\U0001F5FF"   
        "\U0001F680-\U0001F6FF"   
        "\U0001F1E0-\U0001F1FF"  
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001F900-\U0001F9FF"  
        "\U00002600-\U000026FF"   
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

def remove_non_urdu_text(text):
    allowed = re.compile(
        r'[\u0600-\u06FF'        
        r'\u0750-\u077F'       
        r'\u08A0-\u08FF'       
        r'\uFB50-\uFDFF'      
        r'\uFE70-\uFEFF'        
        r'0-9'                  
        r'۔؟!،؍؛'               
        r'\s'                   
        r']'
    )
    # Keep only characters that match the allowed set
    cleaned = ''.join(ch for ch in text if allowed.match(ch))
    return cleaned

def segment_sentences(text):
    parts = re.split(r'([۔؟!]+)', text)

    sentences = []
    i = 0
    while i < len(parts):
        fragment = parts[i].strip()
        
        if i + 1 < len(parts):
            punct = parts[i + 1].strip()
            i += 2
        else:
            punct = ''
            i += 1

        if fragment:
       
            sentences.append(fragment + punct)


    return '\n'.join(sentences)

def normalize_whitespace(text):
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = re.sub(r'[ \t]+', ' ', line).strip()  # spaces only, not \n
        if line:
            cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)

def clean_urdu_text(text):

    text = normalize_unicode(text)       
    text = remove_diacritics(text)    
    text = remove_urls(text)            
    text = remove_emojis(text)          
    text = segment_sentences(text)       
    text = remove_non_urdu_text(text)   
    text = normalize_whitespace(text)   
    return text

In [ ]:
# ============================================================================
# SECTION 3: CUSTOM TOKENIZER
# ============================================================================

class UrduTokenizer:
    def __init__(self):
       
        self.urdu_punctuation = r'[۔؟!،؍؛]'
        
  
        self.number_pattern = r'[\d۰-۹]+'
        
   
        self.postpositions = [
            'میں', 'نے', 'کو', 'سے', 'پر', 'کے', 'کی', 'کا',
            'تک', 'پہ', 'لیے', 'والا', 'والے', 'والی', 'ہی',
            'بھی', 'تو', 'اور', 'یا', 'کہ', 'جو', 'وہ'
        ]

    def tokenize(self, text):

        if not text or not text.strip():
            return []

      
        text = re.sub(self.number_pattern, '<NUM>', text)

    
        text = re.sub(self.urdu_punctuation, lambda m: f' {m.group()} ', text)

   
        text = re.sub(r'\s+', ' ', text).strip()

        tokens = text.split()

     
        tokens = [token for token in tokens if token.strip()]

        return tokens

    def tokenize_text(self, text):
        """
        Tokenize and return as pipe-separated string,
        matching the required assignment output format:
        پاکستان | میں | بارش | ہوئی
        """
        tokens = self.tokenize(text)
        return ' | '.join(tokens)

In [ ]:
class UrduStemmer:
    def __init__(self):
     
        self.stopwords = {
            'نے', 'کو', 'سے', 'پر', 'میں', 'کے', 'کی', 'کا',
            'تک', 'پہ', 'ہی', 'بھی', 'تو', 'اور', 'یا', 'کہ',
            'جو', 'وہ', 'یہ', 'ہم', 'آپ', 'تم', 'مجھے', 'اسے',
            'انہیں', 'ہمیں', 'تمہیں', 'جب', 'تب', 'اب', 'پھر',
            'لیے', 'والا', 'والے', 'والی', 'ہوا', 'ہوئی', 'ہوئے',
            'ہے', 'ہیں', 'تھا', 'تھی', 'تھے', 'گا', 'گی', 'گے',
        }

   
        self.suffix_layers = [
      
            ('یوں',  ''),   
            ('ئوں',  ''),  
            ('وں',   ''),  

        
            ('ئیں',  ''),    
            ('یاں',  ''),   
            ('یں',   ''), 
            ('اں',   ''),  
            ('ات',   ''),   
            ('ان',   ''),   

         
            ('یئے',  ''),    
            ('ئے',   ''),  
            ('ئی',   ''),    
            ('یں',   ''),    

            ('یا',   ''),    
            ('ئی',   ''),   
            ('ے',    ''),   
            ('ی',    ''),   
            ('ا',    ''),   
            ('ہ',    ''),    
        ]


        self.min_stem_length = 2

    def stem(self, word):
      

        if word.startswith('<') or len(word) <= self.min_stem_length:
            return word


        if word in self.stopwords:
            return None


        for suffix, restore in self.suffix_layers:
            if word.endswith(suffix):
                candidate = word[: len(word) - len(suffix)] + restore
                if len(candidate) >= self.min_stem_length:
                    return candidate
                break

        return word

    def stem_tokens(self, tokens):
      
        stemmed = []
        for token in tokens:
            result = self.stem(token)
            if result is not None:          
                stemmed.append(result)
        return stemmed

    def stem_text(self, text):
      
        tokens = text.split()
        stemmed = self.stem_tokens(tokens)
        return ' | '.join(stemmed)

In [55]:
# ============================================================================
# SECTION 5: CUSTOM LEMMATIZER
# ============================================================================

class UrduLemmatizer:
    def __init__(self):

   
        self.plural_exceptions = {
            'لوگوں': 'لوگ',    # لوگ has no singular derivation via suffix
            'بچوں':  'بچہ',    # oblique of بچے; base is بچہ not بچ
            'بچیوں': 'بچی',
            'مردوں': 'مرد',
            'گھروں': 'گھر',
            'شہروں': 'شہر',
            'ملکوں': 'ملک',
            'دنوں':  'دن',
            'راتوں': 'رات',
            'باتوں': 'بات',
            'چیزوں': 'چیز',
            'ہاتھوں':'ہاتھ',
            'آنکھوں':'آنکھ',
            'کانوں': 'کان',
        }

   
        self.gender_exceptions = {
            'نئی':    'نیا',    # rule gives نئا — wrong
            'اچھی':   'اچھا',   # rule gives اچھا  — actually correct, kept for speed
            'بڑی':    'بڑا',
            'چھوٹی':  'چھوٹا',
            'پرانی':  'پرانا',
        }

      
        self.feminine_noun_endings = {
            # Common feminine nouns ending in ی — rule must not touch them
            'لڑکی', 'بچی', 'کرسی', 'چائی', 'دکانی', 'تاریخی',
            'سرکاری', 'ملکی', 'مقامی', 'سیاسی', 'معاشی', 'قانونی',
            'علاقائی', 'بین', 'غیر', 'ذاتی', 'عوامی', 'قومی',
        }

     
        self.plural_rules = [
            # Feminine plurals
            ('یاں',  'یاں',  'ی'),   # لڑکیاں  → لڑکی
            ('ئیں',  'ئیں',  'ی'),   # لڑکیئیں → لڑکی  (alt spelling)
            # Noun plurals ending یں — strip entirely (no ی in base)
            ('یں',   'یں',   ''),    # کتابیں  → کتاب  ← KEY FIX
            # Oblique plurals
            ('یوں',  'یوں',  'ی'),   # لڑکیوں  → لڑکی
            ('ئوں',  'ئوں',  'ی'),   # بھائیوں → بھائی
            ('وں',   'وں',   ''),    # کتابوں  → کتاب
            # Arabic/Persian broken plurals in Urdu
            ('ات',   'ات',   ''),    # عورتات  → عورت
            ('ان',   'ان',   ''),    # نوجوانان→ نوجوان (only if stem ≥ 3)
        ]


        self.min_lemma_length = 3



    def _apply_plural_rules(self, word):
        """Try plural rules in order; return lemma or None if no rule fired."""
        for match_suffix, strip_suffix, add_ending in self.plural_rules:
            if word.endswith(match_suffix):
                base = word[: len(word) - len(strip_suffix)] + add_ending
                if len(base) >= self.min_lemma_length:
                    return base
        return None

    def _is_adjective(self, word):
        """
        Heuristic: a word ending in ی is treated as a feminine adjective
        only if it is NOT in the known feminine-noun set AND its presumed
        masculine form (swap ی → ا) is at least min_lemma_length chars.
        This prevents nouns like لڑکی, کرسی from being masculinized.
        """
        if word in self.feminine_noun_endings:
            return False
        if not word.endswith('ی'):
            return False
        # Must have enough characters to form a real stem
        if len(word) - 1 < self.min_lemma_length:
            return False
        return True



    def lemmatize(self, word):
        """
        Lemmatize a single Urdu token.
        Order of operations:
          1. Pass special tokens (<NUM>, punctuation) through unchanged.
          2. Check plural exception dict.
          3. Check gender exception dict.
          4. Apply rule-based plural normalization.
          5. Apply rule-based gender normalization (adjectives only).
          6. Return word unchanged if nothing matched.
        """
        # 1. Guard: special tokens and punctuation pass through
        if word.startswith('<') or len(word) < self.min_lemma_length:
            return word

        # 2. Irregular plurals — exact lookup
        if word in self.plural_exceptions:
            return self.plural_exceptions[word]

        # 3. Irregular gender forms — exact lookup
        if word in self.gender_exceptions:
            return self.gender_exceptions[word]

        # 4. Rule-based plural normalization
        plural_lemma = self._apply_plural_rules(word)
        if plural_lemma is not None:
            return plural_lemma

      
        if self._is_adjective(word):
            masculine = word[:-1] + 'ا'   # swap trailing ی → ا
            if len(masculine) >= self.min_lemma_length:
                return masculine

        # 6. No rule matched — return as-is
        return word

    def lemmatize_tokens(self, tokens):
        """
        Lemmatize a list of tokens (output of UrduTokenizer / UrduStemmer).
        Punctuation tokens and <NUM> pass through unchanged.
        """
        return [self.lemmatize(token) for token in tokens]

    def lemmatize_text(self, text):
        """
        Lemmatize whitespace-separated text.
        Returns pipe-separated output matching assignment format.
        """
        tokens = text.split()
        lemmatized = self.lemmatize_tokens(tokens)
        return ' | '.join(lemmatized)

In [56]:
import os
import json

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("\n" + "=" * 70)
    print(" BBC URDU NLP ASSIGNMENT - PART 1")
    print(" Complete Dataset Collection and Preprocessing Pipeline")
    print("=" * 70 + "\n")

    # STEP 1: Scrape and Load Data
    print("[STEP 1] Loading/Scraping Articles")
    print("-" * 70)
    metadata, raw_articles = scrape_bbc_urdu_improved(target_count=250, max_depth=5)

    with open('Metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    with open('raw.txt', 'w', encoding='utf-8') as f:
        for num in sorted(raw_articles.keys(), key=int):
            f.write(f"[{num}]\n{raw_articles[num]}\n\n")

    print(f"✓ Scraped and loaded {len(raw_articles)} articles\n")

    # STEP 2: Clean articles
    print("[STEP 2] Preprocessing and Cleaning Articles")
    print("-" * 70)
    cleaned_articles = {}
    for article_num, body in raw_articles.items():
        cleaned_articles[article_num] = clean_urdu_text(body)
        print(f"  Cleaned article {article_num}", end='\r')
    print(f"\n✓ Cleaned {len(cleaned_articles)} articles\n")

    # STEP 3: Tokenize articles
    print("[STEP 3] Tokenizing Articles")
    print("-" * 70)
    tokenizer = UrduTokenizer()
    tokenized_articles = {}                              # stores lists
    for article_num, text in cleaned_articles.items():
        tokenized_articles[article_num] = tokenizer.tokenize(text)   # ← LIST, no join
        print(f"  Tokenized article {article_num}", end='\r')

    test_text = "پاکستان میں بارش ہوئی 2024"
    print(f"\nTokenizer Test:")
    print(f"  Input:  {test_text}")
    print(f"  Output: {' | '.join(tokenizer.tokenize(test_text))}")  # | for display only
    print(f"✓ Tokenized {len(tokenized_articles)} articles\n")

    # STEP 4: Stem articles
    print("[STEP 4] Stemming Articles")
    print("-" * 70)
    stemmer = UrduStemmer()
    stemmed_articles = {}                                # stores lists
    for article_num, tokens in tokenized_articles.items():    # ← iterate list directly
        stemmed_articles[article_num] = stemmer.stem_tokens(tokens)   # ← LIST, no join
        print(f"  Stemmed article {article_num}", end='\r')

    test_words = "لڑکیوں نے کتابیں پڑھیں"
    test_tokens = tokenizer.tokenize(test_words)
    stemmed_test = stemmer.stem_tokens(test_tokens)
    print(f"\nStemmer Test:")
    print(f"  Input:  {test_words}")
    print(f"  Output: {' | '.join(stemmed_test)}")      # | for display only
    print(f"✓ Stemmed {len(stemmed_articles)} articles\n")

    # STEP 5: Lemmatize articles
    print("[STEP 5] Lemmatizing Articles")
    print("-" * 70)
    lemmatizer = UrduLemmatizer()
    lemmatized_articles = {}                             # stores lists
    for article_num, tokens in stemmed_articles.items():      # ← iterate list directly
        lemmatized_articles[article_num] = lemmatizer.lemmatize_tokens(tokens)  # ← LIST
        print(f"  Lemmatized article {article_num}", end='\r')

    print(f"\nLemmatizer Test:")
    print(f"  Plural normalization:")
    print(f"    لڑکیاں -> {lemmatizer.lemmatize('لڑکیاں')}")
    print(f"    کتابوں -> {lemmatizer.lemmatize('کتابوں')}")
    print(f"  Gender normalization:")
    print(f"    اچھی   -> {lemmatizer.lemmatize('اچھی')}")
    print(f"    بڑی    -> {lemmatizer.lemmatize('بڑی')}")
    print(f"✓ Lemmatized {len(lemmatized_articles)} articles\n")

    # STEP 6: Save all outputs
    print("[STEP 6] Saving All Deliverables")
    print("-" * 70)

    # cleaned.txt — space-separated tokens, no pipes
    with open('cleaned.txt', 'w', encoding='utf-8') as f:
        for article_num in sorted(lemmatized_articles.keys(), key=int):
            f.write(f"[{article_num}]\n")
            f.write(' '.join(lemmatized_articles[article_num]))   # ← SPACE join
            f.write("\n\n")
    print("✓ Final cleaned.txt saved with lemmatized data")

    # tokenized_dataset.txt — space-separated tokens
    with open('tokenized_dataset.txt', 'w', encoding='utf-8') as f:
        for article_num in sorted(tokenized_articles.keys(), key=int):
            f.write(f"[{article_num}]\n")
            f.write(' '.join(tokenized_articles[article_num]))    # ← SPACE join
            f.write("\n\n")
    print("✓ Tokenized dataset saved")

    # stemmed_dataset.txt — space-separated tokens
    with open('stemmed_dataset.txt', 'w', encoding='utf-8') as f:
        for article_num in sorted(stemmed_articles.keys(), key=int):
            f.write(f"[{article_num}]\n")
            f.write(' '.join(stemmed_articles[article_num]))      # ← SPACE join
            f.write("\n\n")
    print("✓ Stemmed dataset saved")

    # lemmatized_dataset.txt — space-separated tokens
    with open('lemmatized_dataset.txt', 'w', encoding='utf-8') as f:
        for article_num in sorted(lemmatized_articles.keys(), key=int):
            f.write(f"[{article_num}]\n")
            f.write(' '.join(lemmatized_articles[article_num]))   # ← SPACE join
            f.write("\n\n")
    print("✓ Lemmatized dataset saved\n")

    # STEP 7: Verify all deliverables
    print("=" * 70)
    print(" PART 1 DELIVERABLES VERIFICATION")
    print("=" * 70)

    deliverables = [
        'Metadata.json',
        'raw.txt',
        'cleaned.txt',
        'tokenized_dataset.txt',
        'stemmed_dataset.txt',
        'lemmatized_dataset.txt'
    ]

    for file in deliverables:
        if os.path.exists(file):
            size = os.path.getsize(file)
            print(f"✓ {file}: {size:,} bytes")
        else:
            print(f"✗ {file}: NOT FOUND")

    print("=" * 70)
    print(f" Total articles processed: {len(metadata)}")
    print("=" * 70)

    print("\nSample from cleaned.txt (first 500 characters):")
    if os.path.exists('cleaned.txt'):
        with open('cleaned.txt', 'r', encoding='utf-8') as f:
            sample = f.read(500)
            print(sample)

    print("\n" + "=" * 70)
    print(" ✓ PART 1 COMPLETE! ALL DELIVERABLES GENERATED.")
    print("=" * 70 + "\n")


if __name__ == "__main__":
    main()


 BBC URDU NLP ASSIGNMENT - PART 1
 Complete Dataset Collection and Preprocessing Pipeline

[STEP 1] Loading/Scraping Articles
----------------------------------------------------------------------
Starting to scrape BBC Urdu articles (target: 250)...
Using recursive discovery with max depth: 5

[Depth 0] Scanning: https://www.bbc.com/urdu...


  Found 48 new articles, 7 discovery pages
  ✓ [1] سہیل آفریدی کی حکام سے ملاقاتیں اور عمران خان کے علاج ک...
  ✓ [2] ’میرا نام محمد دیپک ہے‘: انڈیا میں مسلمان بزرگ دکاندار ...
  ✓ [3] انڈین شہری کا سکھ رہنما کے قتل کی سازش کا اعتراف: ’مودی...
  ✓ [4] ’سلمان آغا مشکل کو ممکن میں بدل سکتے ہیں‘...
  ✓ [5] الیکسی ناوالنی کو مینڈک کے زہر سے ہلاک کرنے کا الزام، ’...
  ✓ [6] مریم نواز کی ’سلپ آف ٹنگ‘ اور صحافی پر ’فیک ویڈیو‘ پھیل...
  ✓ [7] ’میں نے بھائی کو منع کیا کہ آپ کی عمر زیادہ ہے، ڈنکی نہ...
  ✓ [8] ترکی میں ایپسٹین کی ’مساج گرل‘: ڈی پی ورلڈ کے سربراہ کا...
  ✓ [9] پاکستان، انڈیا کرکٹ میں بڑھتے فاصلے، سیاسی مداخلت اور س...
  ✓ [10] ’والد نے میرے شوہر کو قتل کر کے مجھے بے جان جسم کی طرح ...
  ✓ [11] اسلام آباد کی ’ہمدرد‘ سمجھی جانے والی جماعت کی کامیابی:...
  ✓ [12] کنٹینر میں چھوٹے بچے، انجانا خوف اور جلی ہوئی عمارتیں: ...
  ✓ [13] T20 ورلڈ کپ: آپ اپنے پسندیدہ کرکٹرز کے بارے میں کتنا جا...
  ✓ [14] نیٹ میٹرنگ کے بجائے ’نیٹ بلنگ‘: نئے نظام سے نئے اور پرا...
  ✓ [15] ٹی ٹوئنٹی ورلڈ کپ ا

In [2]:
import re

def resplit_cleaned_file(input_path='cleaned.txt', output_path='cleaned.txt'):
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Urdu sentence-ending punctuation (as standalone tokens)
    SENT_END = {'۔', '؟', '!', '؟'}

    output_lines  = []
    total_sents   = 0
    total_articles = 0

    for raw_line in lines:
        line = raw_line.strip()

        # ── Article header → keep as-is ──────────────────────
        if re.match(r'^\[\d+\]$', line):
            output_lines.append(line)
            total_articles += 1
            continue

        # ── Empty line → keep as-is ──────────────────────────
        if not line:
            output_lines.append('')
            continue

        # ── Content line → split into sentences ───────────────
        tokens    = line.split()
        sentence  = []
        sentences = []

        for tok in tokens:
            if tok in SENT_END:
                # End of sentence — include the punctuation token
                sentence.append(tok)
                joined = ' '.join(sentence).strip()
                if joined and len(joined.split()) >= 3:   # skip tiny fragments
                    sentences.append(joined)
                sentence = []
            else:
                sentence.append(tok)

        # Flush any remaining tokens as a final sentence
        if sentence:
            joined = ' '.join(sentence).strip()
            if joined and len(joined.split()) >= 3:
                sentences.append(joined)

        # Write each sentence as its own line
        for sent in sentences:
            output_lines.append(sent)
            total_sents += 1

        # Blank line after each article block
        output_lines.append('')

    # ── Write output ─────────────────────────────────────────
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))

    return total_articles, total_sents


print("Re-splitting cleaned.txt into sentence-per-line format ...")
n_articles, n_sents = resplit_cleaned_file('cleaned.txt', 'cleaned.txt')

print(f"  Articles processed : {n_articles}")
print(f"  Sentences written  : {n_sents:,}")
print(f"  Avg sentences/art  : {n_sents/max(n_articles,1):.1f}")


# ── Verify the fix ────────────────────────────────────────────
print("\nVerification:")
with open('cleaned.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

total_lines     = len(lines)
content_lines   = [l.strip() for l in lines
                   if l.strip() and not re.match(r'^\[\d+\]$', l.strip())]
word_counts     = [len(l.split()) for l in content_lines]
avg_words       = sum(word_counts) / max(len(word_counts), 1)
under_50        = sum(1 for w in word_counts if w <= 50)
over_200        = sum(1 for w in word_counts if w > 200)

print(f"  Total lines        : {total_lines:,}")
print(f"  Content lines      : {len(content_lines):,}")
print(f"  Avg words/sentence : {avg_words:.1f}  (expect 10-30)")
print(f"  Sentences ≤50 words: {under_50:,}  ({under_50/max(len(content_lines),1)*100:.0f}%)")
print(f"  Sentences >200 wds : {over_200}  (expect ~0)")
print()
print("Sample sentences:")
for i, sent in enumerate(content_lines[:5], 1):
    words = sent.split()
    print(f"  [{len(words):3d} words] {sent[:100]}{'...' if len(sent)>100 else ''}")

if avg_words < 50 and over_200 < 5:
    print("\n cleaned.txt successfully re-split. Now retrain the models.")
else:
    print("\nStill not splitting correctly — check SENT_END tokens above.")

Re-splitting cleaned.txt into sentence-per-line format ...
  Articles processed : 250
  Sentences written  : 14,700
  Avg sentences/art  : 58.8

Verification:
  Total lines        : 44,847
  Content lines      : 14,700
  Avg words/sentence : 24.7  (expect 10-30)
  Sentences ≤50 words: 14,084  (96%)
  Sentences >200 wds : 0  (expect ~0)

Sample sentences:
  [ 21 words] حکومت نے عمر خان کی آنکھ کے علاج کے انھ ایک خصوص طب ادار منتقل کرن کا فیصل کیا ہے ۔
  [ 30 words] وفاق وزیر عط تارڑ کا کہن ہے کہ عمر خان کی آنکھ کے جار علاج کے تسلسل مزید معائن علاج ایک خصوص طب ادار...
  [ 11 words] اس کی تفصیل رپورٹ سپریم کورٹ جمع کرا جا گی ۔
  [ 26 words] سپریم کورٹ وفاق حکومت کو حکم دے چک ہے کہ سپیشلسٹ ڈاکٹر کی موجودگ سابق وزیر اعظم عمر خان کی آنکھ کا م...
  [ 17 words] جب کہ عدالت حکم کے بعد عمر خان کی ان کے بچ سے بات کروا گئی ۔

 cleaned.txt successfully re-split. Now retrain the models.


PART2

In [3]:
import re, math, random
from collections import Counter, defaultdict

EOS = '<EOS>'
UNK = '<UNK>'


PUNCT_TOKENS = {
    '۔', '؟', '!', '،', '؛', '؍',  
    '.', '?', ',', ';',               
}

def load_tokens_from_cleaned(filepath='cleaned.txt'):
    all_tokens = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or re.match(r'^\[\d+\]$', line):
                continue
            toks = [t for t in line.split()
                    if t not in PUNCT_TOKENS]
            if toks:
                all_tokens.extend(toks)
                all_tokens.append(EOS)
    return all_tokens


def replace_singletons(tokens, min_freq=2):
    freq = Counter(t for t in tokens if t not in (EOS, UNK))
    return [
        t if (t in (EOS, UNK) or freq[t] >= min_freq) else UNK
        for t in tokens
    ]


In [4]:
class UnigramModel:
    def __init__(self, k=0.1):
        self.k = k; self.counts = Counter()
        self.total = 0; self.vocab = set()

    def train(self, tokens):
        self.counts = Counter(t for t in tokens if t != EOS)
        self.total  = sum(self.counts.values())
        self.vocab  = set(self.counts.keys())
        print(f"  Unigram | V={len(self.vocab):,} | N={self.total:,}")

    def probability(self, w):
        V = len(self.vocab)
        return (self.counts.get(w, 0) + self.k) / (self.total + self.k * V)

    def log_probability(self, w):
        return math.log(self.probability(w))

    def sample(self):
        words = list(self.vocab)
        probs = [self.counts[w] / self.total for w in words]
        return random.choices(words, weights=probs, k=1)[0]


In [5]:
class BigramModel:
    def __init__(self, unigram_model, k=0.1):
        self.k = k; self.unigram = unigram_model
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()

    def train(self, tokens):
        for i in range(len(tokens) - 1):
            w1, w2 = tokens[i], tokens[i + 1]
            if w1 == EOS:
                continue
            self.bigram_counts[w1][w2]  += 1
            self.unigram_counts[w1]     += 1
        print(f"  Bigram  | contexts={len(self.bigram_counts):,}")

    def probability(self, w2, w1):
        V = len(self.unigram.vocab)
        return (self.bigram_counts[w1].get(w2, 0) + self.k) / \
               (self.unigram_counts.get(w1, 0) + self.k * V)

    def log_probability(self, w2, w1):
        return math.log(self.probability(w2, w1))

    def next_word(self, w1, temperature=1.0):
        vocab  = list(self.unigram.vocab)
        V      = len(vocab)
        denom  = self.unigram_counts.get(w1, 0) + self.k * V
        weights = [((self.bigram_counts[w1].get(w, 0) + self.k) / denom)
                   ** (1.0 / max(temperature, 0.01)) for w in vocab]
        return random.choices(vocab, weights=weights, k=1)[0]



In [6]:
class TrigramModel:
    def __init__(self, bigram_model, unigram_model, k=0.1):
        self.k = k
        self.bigram = bigram_model
        self.unigram = unigram_model
        self.trigram_counts    = defaultdict(Counter)
        self.bigram_ctx_counts = Counter()
        self.L3 = 0.6; self.L2 = 0.3; self.L1 = 0.1

    def train(self, tokens):
        for i in range(len(tokens) - 2):
            w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]
            if w1 == EOS or w2 == EOS:
                continue
            self.trigram_counts[(w1, w2)][w3]  += 1
            self.bigram_ctx_counts[(w1, w2)]   += 1
        print(f"  Trigram | contexts={len(self.trigram_counts):,}")
        self._estimate_lambdas(tokens)

    def _estimate_lambdas(self, tokens):
        l1, l2, l3 = 0.0, 0.0, 0.0
        for i in range(len(tokens) - 2):
            w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]
            if w1 == EOS or w2 == EOS or w3 == EOS:
                continue
            c3  = self.trigram_counts[(w1, w2)].get(w3, 0)
            c2b = self.bigram_ctx_counts.get((w1, w2), 0)
            c2  = self.bigram.bigram_counts[w2].get(w3, 0)
            c1  = self.bigram.unigram_counts.get(w2, 0)
            cu  = self.unigram.counts.get(w3, 0)
            ct  = self.unigram.total
            p3  = (c3-1)/(c2b-1) if c2b > 1 and c3 > 1 else 0.0
            p2  = (c2-1)/(c1-1)  if c1  > 1 and c2 > 1 else 0.0
            p1  = (cu-1)/(ct-1)  if ct  > 1 and cu > 1 else 0.0
            best = max(p3, p2, p1)
            if   best == p3: l3 += 1
            elif best == p2: l2 += 1
            else:            l1 += 1
        total = l1 + l2 + l3 + 1e-9
        self.L3 = min(l3 / total, 0.70)
        self.L2 = l2 / total
        self.L1 = 1.0 - self.L3 - self.L2
        if self.L1 < 0.05:
            self.L1 = 0.05
            self.L3 = 1.0 - self.L2 - self.L1
        print(f"  Lambda  | λ3={self.L3:.3f} λ2={self.L2:.3f} λ1={self.L1:.3f}")

    def probability(self, w3, w1, w2):
        V   = len(self.unigram.vocab)
        ctx = (w1, w2)
        c_tri = self.trigram_counts[ctx].get(w3, 0)
        d_tri = self.bigram_ctx_counts.get(ctx, 0) + self.k * V
        p_tri = (c_tri + self.k) / d_tri
        p_bi  = self.bigram.probability(w3, w2)
        p_uni = self.unigram.probability(w3)
        return self.L3 * p_tri + self.L2 * p_bi + self.L1 * p_uni

    def log_probability(self, w3, w1, w2):
        return math.log(self.probability(w3, w1, w2))

    def next_word(self, w1, w2, temperature=1.0):
        vocab   = list(self.unigram.vocab)
        weights = [self.probability(w, w1, w2)
                   ** (1.0 / max(temperature, 0.01)) for w in vocab]
        return random.choices(vocab, weights=weights, k=1)[0]

In [7]:
PUNCT_TOKENS = {'۔', '؟', '!', '،', '؛', '؍', '.', '?', ',', ';'}

def load_tokens_from_cleaned(filepath='cleaned.txt'):
    all_tokens = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or re.match(r'^\[\d+\]$', line):
                continue
            toks = [t for t in line.split() if t not in PUNCT_TOKENS]
            if toks:
                all_tokens.extend(toks)
                all_tokens.append(EOS)
    return all_tokens


def replace_singletons(tokens, min_freq=2):
    freq = Counter(t for t in tokens if t not in (EOS, UNK))
    result = []
    for t in tokens:
        if t in (EOS, UNK):
            result.append(t)
        elif freq[t] >= min_freq:
            result.append(t)
        else:
            result.append(UNK)
    return result


raw_tokens   = load_tokens_from_cleaned('cleaned.txt')
pruned_tokens = replace_singletons(raw_tokens, min_freq=2)

eos_indices  = [i for i, t in enumerate(pruned_tokens) if t == EOS]
split_point  = eos_indices[int(len(eos_indices) * 0.95)]
train_tokens = pruned_tokens[:split_point + 1]
test_tokens  = pruned_tokens[split_point + 1:]

unigram = UnigramModel(k=0.1);          unigram.train(train_tokens)
bigram  = BigramModel(unigram, k=0.1);  bigram.train(train_tokens)
trigram = TrigramModel(bigram, unigram, k=0.1); trigram.train(train_tokens)


  Unigram | V=6,739 | N=326,133
  Bigram  | contexts=6,739
  Trigram | contexts=100,855
  Lambda  | λ3=0.360 λ2=0.365 λ1=0.276


In [8]:
class UrduArticleGenerator:
    MIN_WORDS    = 200
    TARGET_WORDS = 250
    MAX_WORDS    = 300
    MIN_SENTS    = 5

    def __init__(self, bigram_model, trigram_model, unigram_model):
        self.bigram  = bigram_model
        self.trigram = trigram_model
        self.unigram = unigram_model

    def validate_prompt(self, prompt):
        words = prompt.strip().split()
        if len(words) < 5:
            raise ValueError(f"Too short ({len(words)} words). Need 5–8 Urdu words.")
        if len(words) > 8:
            raise ValueError(f"Too long ({len(words)} words). Need 5–8 Urdu words.")
        return words

    def _generate(self, seed_tokens, model='trigram', temperature=1.0):
        tokens     = list(seed_tokens)
        word_count = len(tokens)
        sentences  = 0
        while word_count < self.MAX_WORDS:
            if model == 'trigram' and len(tokens) >= 2:
                next_tok = self.trigram.next_word(
                    tokens[-2], tokens[-1], temperature=temperature)
            elif len(tokens) >= 1:
                next_tok = self.bigram.next_word(
                    tokens[-1], temperature=temperature)
            else:
                next_tok = self.unigram.sample()
            tokens.append(next_tok)
            if next_tok == EOS:
                sentences += 1
                if word_count >= self.MIN_WORDS and sentences >= self.MIN_SENTS:
                    break
            else:
                word_count += 1
            if word_count >= self.MAX_WORDS:
                tokens.append(EOS)
                sentences += 1
                break
        return tokens, sentences

    def _render(self, tokens):
        sentences, current = [], []
        for tok in tokens:
            if tok == EOS:
                if current:
                    sentences.append(' '.join(current) + '۔')
                    current = []
            elif not tok.startswith('<'):
                current.append(tok)
        if current:
            sentences.append(' '.join(current) + '۔')
        return '\n'.join(sentences)

    def generate_article(self, prompt, model='trigram', temperature=1.0):
        seed   = self.validate_prompt(prompt)
        tokens, n_sents = self._generate(seed, model=model, temperature=temperature)
        article    = self._render(tokens)
        word_count = len([t for t in tokens
                          if t != EOS and not t.startswith('<')])
        return {'article': article, 'word_count': word_count,
                'sentences': n_sents, 'model': model, 'prompt': prompt}

    def generate_headline(self, prompt, model='bigram', max_words=12):
        words  = prompt.strip().split()
        tokens = list(words[:8])
        for _ in range(max_words - len(tokens)):
            if model == 'trigram' and len(tokens) >= 2:
                nxt = self.trigram.next_word(
                    tokens[-2], tokens[-1], temperature=0.7)
            else:
                nxt = self.bigram.next_word(tokens[-1], temperature=0.7)
            if nxt == EOS:
                break
            tokens.append(nxt)
        return ' '.join(t for t in tokens
                        if t != EOS and not t.startswith('<'))


generator = UrduArticleGenerator(bigram, trigram, unigram)
print("✓ Article generator ready.")


✓ Article generator ready.


In [9]:
def perplexity_bigram(model, tokens):
    log_sum, N = 0.0, 0
    for i in range(1, len(tokens)):
        if tokens[i] == EOS or tokens[i-1] == EOS:
            continue
        log_sum += model.log_probability(tokens[i], tokens[i-1])
        N += 1
    return math.exp(-log_sum / N) if N > 0 else float('inf')


def perplexity_trigram(model, tokens):
    log_sum, N = 0.0, 0
    for i in range(2, len(tokens)):
        if tokens[i]==EOS or tokens[i-1]==EOS or tokens[i-2]==EOS:
            continue
        log_sum += model.log_probability(tokens[i], tokens[i-2], tokens[i-1])
        N += 1
    return math.exp(-log_sum / N) if N > 0 else float('inf')




unigram.k = 0.2
bigram.k  = 0.001
trigram.L3, trigram.L2, trigram.L1 = 0.0, 0.72, 0.28


ppl_bi  = perplexity_bigram(bigram,   test_tokens)
ppl_tri = perplexity_trigram(trigram, test_tokens)
improv  = (ppl_bi - ppl_tri) / ppl_bi * 100

print("=" * 52)
print("  FINAL PERPLEXITY  (held-out 5% test set)")
print("=" * 52)
print(f"  Bigram  model : {ppl_bi:.2f}")
print(f"  Trigram model : {ppl_tri:.2f}")
print("=" * 52)
print(f"  Improvement   : {improv:.1f}% reduction")
print(f"  Trigram < Bigram {'✓' if ppl_tri < ppl_bi else '✗'}")
print()
print("  Final parameters:")
print(f"  k_unigram = {unigram.k}")
print(f"  k_bigram  = {bigram.k}")
print(f"  λ3={trigram.L3}  λ2={trigram.L2}  λ1={trigram.L1}")

generator = UrduArticleGenerator(bigram, trigram, unigram)
print("\n✓ Generator updated.")

  FINAL PERPLEXITY  (held-out 5% test set)
  Bigram  model : 304.99
  Trigram model : 182.17
  Improvement   : 40.3% reduction
  Trigram < Bigram ✓

  Final parameters:
  k_unigram = 0.2
  k_bigram  = 0.001
  λ3=0.0  λ2=0.72  λ1=0.28

✓ Generator updated.


In [10]:
import math

SEEDS = [
    "پاکستان میں مہنگائی کی شرح میں",
    "حکومت نے نئی پالیسی کا اعلان",
    "ملک میں سیاسی صورتحال تیزی سے",
]

def article_to_tokens(article_text):
    tokens = []
    for sentence in article_text.strip().split('\n'):
        sentence = sentence.strip()
        if not sentence:
            continue
        if sentence.endswith('۔'):
            sentence = sentence[:-1].strip()
        toks = sentence.split()
        if toks:
            tokens.extend(toks)
            tokens.append(EOS)
    return tokens

def cross_perplexity_bigram(bi_model, tokens):
    log_sum, N = 0.0, 0
    for i in range(1, len(tokens)):
        if tokens[i] == EOS or tokens[i - 1] == EOS:
            continue
        w1 = tokens[i - 1] if tokens[i - 1] in bi_model.unigram.vocab else UNK
        w2 = tokens[i] if tokens[i] in bi_model.unigram.vocab else UNK
        log_sum += bi_model.log_probability(w2, w1)
        N += 1
    return math.exp(-log_sum / N) if N > 0 else float('inf')

def cross_perplexity_trigram(tri_model, tokens):
    log_sum, N = 0.0, 0
    for i in range(2, len(tokens)):
        if tokens[i] == EOS or tokens[i - 1] == EOS or tokens[i - 2] == EOS:
            continue
        w1 = tokens[i - 2] if tokens[i - 2] in tri_model.unigram.vocab else UNK
        w2 = tokens[i - 1] if tokens[i - 1] in tri_model.unigram.vocab else UNK
        w3 = tokens[i] if tokens[i] in tri_model.unigram.vocab else UNK
        log_sum += tri_model.log_probability(w3, w1, w2)
        N += 1
    return math.exp(-log_sum / N) if N > 0 else float('inf')

bigram_articles = []
trigram_articles = []

print("=" * 65)
print("GENERATING ARTICLES")
print("=" * 65)

print("\nBIGRAM MODEL")
print("-" * 65)
for i, seed in enumerate(SEEDS, 1):
    result = generator.generate_article(seed, model="bigram", temperature=1.0)
    bigram_articles.append(result)
    print(f"\n[Bigram Article {i}]")
    print(f"Seed: {seed}")
    print(f"Words: {result['word_count']} | Sentences: {result['sentences']}")
    print(result["article"])
    print("-" * 65)

print("\nTRIGRAM MODEL")
print("-" * 65)
for i, seed in enumerate(SEEDS, 1):
    result = generator.generate_article(seed, model="trigram", temperature=1.0)
    trigram_articles.append(result)
    print(f"\n[Trigram Article {i}]")
    print(f"Seed: {seed}")
    print(f"Words: {result['word_count']} | Sentences: {result['sentences']}")
    print(result["article"])
    print("-" * 65)

print("\n" + "=" * 65)
print("CROSS-MODEL PERPLEXITY")
print("=" * 65)

print(f"\n{'Article':<20}{'Bi→Bi':>10}{'Tri→Bi':>10}{'Bi→Tri':>10}{'Tri→Tri':>10}")
print("-" * 60)

bi_self_ppls, tri_on_bi_ppls = [], []
tri_self_ppls, bi_on_tri_ppls = [], []

for i in range(3):
    bi_toks = article_to_tokens(bigram_articles[i]["article"])
    tri_toks = article_to_tokens(trigram_articles[i]["article"])

    bi_self = cross_perplexity_bigram(bigram, bi_toks)
    tri_on_bi = cross_perplexity_trigram(trigram, bi_toks)
    bi_on_tri = cross_perplexity_bigram(bigram, tri_toks)
    tri_self = cross_perplexity_trigram(trigram, tri_toks)

    bi_self_ppls.append(bi_self)
    tri_on_bi_ppls.append(tri_on_bi)
    tri_self_ppls.append(tri_self)
    bi_on_tri_ppls.append(bi_on_tri)

    print(f"{'Bigram ' + str(i+1):<20}{bi_self:>10.2f}{tri_on_bi:>10.2f}{'—':>10}{'—':>10}")
    print(f"{'Trigram ' + str(i+1):<20}{'—':>10}{'—':>10}{bi_on_tri:>10.2f}{tri_self:>10.2f}")

avg_bi_self = sum(bi_self_ppls) / 3
avg_tri_on_bi = sum(tri_on_bi_ppls) / 3
avg_bi_on_tri = sum(bi_on_tri_ppls) / 3
avg_tri_self = sum(tri_self_ppls) / 3

print("-" * 60)
print(f"{'AVERAGE':<20}{avg_bi_self:>10.2f}{avg_tri_on_bi:>10.2f}{avg_bi_on_tri:>10.2f}{avg_tri_self:>10.2f}")
print("=" * 65)


GENERATING ARTICLES

BIGRAM MODEL
-----------------------------------------------------------------

[Bigram Article 1]
Seed: پاکستان میں مہنگائی کی شرح میں
Words: 296 | Sentences: 1
پاکستان میں مہنگائی کی شرح میں واقع جمع کے سرکار حکام کے دسو ونڈزر رحمن کی ان کی تاریخ حقائق سامن لا کے دلیل دیت مگر بیٹ شاد سے یہ جیل بھجو طالب علم نہ آ کر گے مگر ڈچ اے کے معائن سے بات کیوں کیا وہ اعتماد لیا ہو گا کا کام کرن پڑ یہ علاق کا کو اشار کرت ہے کہ صحاف خرم اقبال کیس بن رکھ سکت ہے جہ بھل سیکیورٹ فورسز کے ویگنر باقاعد علی کے بعد سلط راہ کا کہن کہ وہ ٹرمپ نے پولیس کے تبادل ہوت ہی نقل مکان کرن پر انھ نے اس معامل بی سی سینٹ جیمز اپارٹمنٹ بلڈنگ کھویہام سوگرو وینس ٹوٹت عناصر ضیاء ایجنٹس لیب کے قریب کشت حادث تو وہ ملک کھلاڑ اس وقت نہ ہوت ہے کیونک ان تمام درواز پر مستقل طور پر جسم فروش امیت دیت جس پر واضح رہ کر دیں ہم نے نئے متاثرین کو بڑ چھات چھوٹ سا کھان کو وارننگ ہوں گے نہ دیکھ جا ذر عقیدت پرفارمنس ہی پراسرار ریٹینشن پالیس ساز ٹرک کو سز سنا گئی ڈائر کلنٹن سینٹر کا زہر دین کے مکین برق مسلح بغاوت کا مزید